# Create gold tables (joins, aggregations) to help us create dashboards for analytics

## Create a few gold tables that will help us answer business questions using all three of the `users`, `transactions_enhanced`, and `cards` tables.

### List of business questions
- Which day(s) of the week sees the highest number of fraudulent transactions?
- What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?
- Which users have the largest number of flagged (“is_fraud = true”) transactions?
- Are there any users showing a sharp rise in transaction amount compared to their weekly average?
- Which merchant categories exhibit the highest fraud rate?
- Are there specific merchants with unusually high fraud volume?
- How does fraud distribution vary by time of day (morning vs night)?
- What’s the average transaction amount for fraud vs non-fraud transactions?
- Which device types are most associated with fraudulent transactions?
- Are there instances where the same device ID is used by multiple users flagged for fraud?
- What are the total monetary losses due to fraud each day?
- How many unique users commit fraudulent transactions per week?
- Do fraud patterns show seasonal or monthly spikes?
- How has user behavior changed before versus after a fraudulent event?
- Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

We don't need all the columns from each table, so we'll do one join to create one gold table with all of the necessary columns
<br>
**Transactions**
- fraud_label
- date
- client_id
- merchant_id
- amount
- mcc
<br>

**Users**
- user_id

**Cards**
- card_type
- card_brand
- card_id (sensitive, usually should not include)

In [0]:
%sql
-- setting the catalog
USE jarvis_training_catalog.silver;

In [0]:
%sql
DROP TABLE IF EXISTS jarvis_training_catalog.gold.gold_enhanced_table;

In [0]:
query = '''
SELECT 
C.card_type,
C.id AS card_id,
C.card_brand,
T.transaction_id,
T.amount,
T.date,
EXTRACT(MONTH FROM T.date) AS month,
DATE_FORMAT(T.date, 'EEEE') AS day_of_week,
CONCAT(EXTRACT(HOUR FROM T.date), ':', EXTRACT(MINUTE FROM T.date)) AS time_of_day,
T.mcc,
CASE
    WHEN T.fraud_label = 'Yes' THEN true
    WHEN T.fraud_label = 'No'  THEN false
    ELSE NULL
  END AS is_fraud,
T.client_id,
T.merchant_id,
U.id AS user_id
FROM
cards C
JOIN 
transactions_enhanced T 
ON C.id = T.card_id
JOIN
users U
ON T.client_id = U.id
'''

gold_df = spark.sql(query)  
display(gold_df.head(5))

card_type,card_id,card_brand,transaction_id,amount,date,month,day_of_week,time_of_day,mcc,is_fraud,client_id,merchant_id,user_id
Debit,5016,Visa,19642741,13.34,2017-06-11T12:24:00.000Z,6,Sunday,12:24,5300,false,639,60569,639
Debit,289,Visa,19643011,6.25,2017-06-11T13:10:00.000Z,6,Sunday,13:10,5541,false,1788,22204,1788
Debit,2500,Visa,19645481,28.64,2017-06-12T05:30:00.000Z,6,Monday,5:30,4784,false,1041,39021,1041
Credit,4877,Visa,19645851,126.07,2017-06-12T07:29:00.000Z,6,Monday,7:29,5912,false,639,14748,639
Credit,4387,Discover,19645930,2.29,2017-06-12T07:52:00.000Z,6,Monday,7:52,5499,false,1067,43293,1067


In [0]:
gold_df.write.mode('overwrite').saveAsTable('jarvis_training_catalog.gold.gold_enhanced_table')